# Khai báo các thư viện cần dùng

In [8]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import explained_variance_score, r2_score
from sklearn.impute import SimpleImputer


# Phần 1. Tiền xử lý dữ liệu

In [9]:
file_path = '../data/Life Expectancy Data.csv'  
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()
df.head()

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,...,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,...,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5


In [10]:
missing_before = df.isna().sum()
missing_count = missing_before.sum()
if df['Life expectancy'].isna().sum() > 0:
    target_mean = df['Life expectancy'].mean()
    target_imputer = SimpleImputer(strategy='mean')
    df['Life expectancy'] = target_imputer.fit_transform(df[['Life expectancy']])
    print(f"Đã impute 'Life expectancy' bằng mean = {target_mean:.2f}")

numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
numeric_cols.remove('Life expectancy')
numeric_cols.remove('Year')

imputer_num = SimpleImputer(strategy='mean')
df[numeric_cols] = imputer_num.fit_transform(df[numeric_cols])
print(f"Đã impute {len(numeric_cols)} features numeric bằng mean")

missing_after = df.isnull().sum().sum()
print(f"\nMissing values sau xử lý: {missing_after}")
print(f"Đã xử lý thành công {missing_count - missing_after} missing values!")

Đã impute 'Life expectancy' bằng mean = 69.22
Đã impute 18 features numeric bằng mean

Missing values sau xử lý: 0
Đã xử lý thành công 2563 missing values!


In [11]:
df = df.drop('Country', axis=1)

print("\n One-Hot Encoding cho 'Status' (drop_first=True để tránh multicollinearity)")
print(f"Trước encoding: Status có {df['Status'].nunique()} unique values")
df = pd.get_dummies(df, columns=['Status'], drop_first=True)
print(f"Sau encoding: Tạo cột 'Status_Developing' (0=Developed, 1=Developing)")


 One-Hot Encoding cho 'Status' (drop_first=True để tránh multicollinearity)
Trước encoding: Status có 2 unique values
Sau encoding: Tạo cột 'Status_Developing' (0=Developed, 1=Developing)


In [12]:
X = df.drop(['Life expectancy'], axis=1)
y = df['Life expectancy']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Phần 2. Phân cụm